# I2SB method comparison

Point `RUNS` at a list of trained configs, run every cell. Each net samples the **same** validation
batch through the reverse bridge, and the result is two figures: slices **with** tumor and slices
**without**, methods side by side.

No metrics. If you want numbers, `training/i2sb.py` already logs them every validation epoch.

## Works on both datasets

Nothing here is dataset-specific. The loader is built from each config's own `data` block and
dispatched by its `name` field, and the NYUMets configs use the same `"i2sb"` loader as BraTS
(`datasets/NYUMets/longitudinal_register.py` says so explicitly). So switching datasets means
swapping the `RUNS` list and nothing else.

## What this replaces

`i2sb_sample.ipynb`, `compare_methods.ipynb`, `compare_i2sb_sweep.ipynb`,
`latent_i2sb_sample.ipynb` and `inspect_latent_tau.ipynb` are in `notebooks/archive/`. The one
capability that did not carry over is `compare_i2sb_sweep`'s per-step / teacher-forced
diagnostics -- pull it back out of the archive if you want those again.

Still current and NOT replaced: `inspect_beta_max.ipynb` (schedule sizing),
`inspect_percentile_norm.ipynb` (normalization), `inspect_i2sb_tau.ipynb` (tau selection).

In [ ]:
import os, sys, json, gc, time
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# ---- the runs to compare: (label, path to that run's config) --------------------------------
# Prefer <save_dir>/config.json -- train.py writes it at launch, so it is what the checkpoint was
# ACTUALLY trained with. The file in config/ works too, but only if you have not edited it since.
RUNS = [
    ("SBCDLNet",  "trained_nets/brats/I2SB_SBCDLNet_T1ce_from_T1_medmad/config.json"),
    ("CDLNet",    "trained_nets/brats/I2SB_CDLNet_T1ce_from_T1/config.json"),
    ("UNet T1",   "trained_nets/brats/I2SB_Unet_T1ce_from_T1/config.json"),
    ("UNet all",  "trained_nets/brats/I2SB_Unet_T1ce_from_all/config.json"),
]

# NYUMets -- same notebook, only this list changes:
# RUNS = [
#     ("SBCDLNet", "trained_nets/nyumets/I2SB_SBCDLNet_NYUMets_CT1_from_all/config.json"),
#     ("UNet all", "trained_nets/nyumets/I2SB_Unet_NYUMets_CT1_from_all/config.json"),
# ]

SPLIT   = "val"
NFE     = 50        # reverse steps; None = each config's own val_nfe
BATCH   = 16        # slices to draw from the split
N_SHOW  = 4         # slices to display per figure
SEED    = 0

# Keep only the middle CENTER_FRAC of each volume; 1.0 = every stored slice. The first and last
# slices of a BraTS / NYUMets volume are mostly background with a sliver of brain -- every method
# reconstructs them identically and they waste the figure. This is a fraction of each SUBJECT's
# own slice count, not a global index cut, because volumes differ in length.
CENTER_FRAC = 0.4
USE_EMA = True      # sample with the EMA shadow when the run saved one

# Fixed display window, NOT per-image percentiles. med/MAD normalization with scales=3 puts the
# brain in roughly [-1, 1], so this is the natural window AND it is the same for every slice and
# every method -- per-image windowing rescales each panel to its own contents, which hides exactly
# the failure you are looking for: a washed-out or low-contrast prediction gets stretched back to
# full range and looks fine next to a good one.
VMIN, VMAX = -1.0, 1.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("repo:", REPO_ROOT, "| device:", device, "|", len(RUNS), "runs")

## One shared batch

Every method must see byte-identical `x0`/`x1`, so the batch is built **once** from the first run's
data block and the rest are checked against it.

`cond_idx` is the deliberate exception -- comparing a T1-only net against an all-contrast net is a
comparison worth making. The loader is built with the *union* of every run's `cond_idx` and each
net is handed the channels it was trained on, sliced out of that one stack.

**Central slices only.** `CENTER_FRAC` keeps the middle band of every volume and drops the end
slices, which are mostly background and reconstruct identically under any method. The cut is per
subject, not a global slice index, so volumes of different lengths contribute the same anatomical
band. The printed brain coverage tells you whether the band is tight enough.

In [ ]:
import datasets                                  # side-effect: registers the loaders
from datasets.registry import build_loader

cfgs = {}
for lbl, p in RUNS:
    if not os.path.exists(p):
        raise FileNotFoundError(f"{lbl!r}: no config at {p}")
    with open(p) as f:
        cfgs[lbl] = json.load(f)

# ---- anything that changes the stored image itself must agree across runs, or x0/x1 are not even
# the same pictures and the panels below would be comparing different data, not different methods.
base = cfgs[RUNS[0][0]]["data"][SPLIT]
MUST_MATCH = ("name", "root", "x0_idx", "x1_idx", "image_key", "scales")
for lbl, _ in RUNS[1:]:
    d = cfgs[lbl]["data"][SPLIT]
    bad = {k: (base.get(k), d.get(k)) for k in MUST_MATCH if base.get(k) != d.get(k)}
    if bad:
        raise ValueError(
            f"{lbl!r} was trained on different data than {RUNS[0][0]!r}: {bad}\n"
            f"These cannot share a batch. Compare them in separate passes of this notebook.")

UNION = sorted({c for lbl, _ in RUNS for c in cfgs[lbl]["data"][SPLIT].get("cond_idx", [])})

data_cfg = dict(base)
data_cfg.update(cond_idx=UNION, num_workers=0, batch_size=BATCH,
                crop_size=None, center_crop=None, random_flips=False)
# BraTS: gives the exact enhancing-tumor mask (and DROPS subjects whose h5 has no 'et').
# NYUMets: preprocessing/nyumets_h5.py skips segmentation files, so there is none -- its loader
# ignores this key and the enhancement fallback below takes over.
data_cfg["et_mask"] = True

try:
    loader = build_loader(data_cfg, shuffle=True, drop_last=False)
except (RuntimeError, KeyError) as e:
    print(f"[info] no ET masks ({type(e).__name__}) -- falling back to enhancement ranking")
    data_cfg["et_mask"] = False
    loader = build_loader(data_cfg, shuffle=True, drop_last=False)

# ---- restrict to central slices --------------------------------------------------------------
# Both loaders flatten (subject, slice) into one index and expose `file_id` / `local`, so the
# middle band is a filter on `local`. Applied PER SUBJECT: volumes differ in length, so one global
# slice-index cut would take a different anatomical band out of each one.
from torch.utils.data import DataLoader, Subset

ds = loader.dataset
if CENTER_FRAC < 1.0:
    n_per_file = np.bincount(ds.file_id)
    pos = ds.local / np.maximum(n_per_file[ds.file_id] - 1, 1)     # 0 = first slice, 1 = last
    keep = np.flatnonzero(np.abs(pos - 0.5) <= CENTER_FRAC / 2)
    if keep.size == 0:
        raise ValueError(f"CENTER_FRAC={CENTER_FRAC} kept no slices out of {len(ds)}")
    if keep.size < BATCH:
        print(f"[warn] only {keep.size} slices pass the filter -- the batch will be short")
    print(f"central {CENTER_FRAC:.0%} of each volume: {keep.size}/{len(ds)} slices "
          f"across {n_per_file.size} subjects")
    loader = DataLoader(Subset(ds, keep.tolist()), batch_size=BATCH, shuffle=True,
                        num_workers=0, drop_last=False)

torch.manual_seed(SEED)
batch = next(iter(loader))
x0, x1, cond, mask = [t.to(device) for t in batch[:4]]
et = batch[4].to(device) if len(batch) > 4 else None

# ---- which slices have tumor? ---------------------------------------------------------------
brain = mask > 0.5
if et is not None and float(et.sum()) > 0:
    score = et.flatten(1).sum(1)                              # ET voxels per slice -- a label
    how = "ET mask"
else:
    # No segmentation: rank by the enhancement itself, the brightest T1ce - T1 difference inside
    # the brain. This is a PROXY, not a label -- it finds enhancement, which is what a met looks
    # like, but it will also rank vessels and any residual mis-normalization highly. Read the
    # "without tumor" figure as "least enhancement in this batch", not "confirmed healthy".
    enh = torch.where(brain, x0 - x1, torch.full_like(x0, -1e9))
    score = enh.flatten(1).quantile(0.999, dim=1)
    how = "enhancement proxy (no segmentation in this dataset)"

order = torch.argsort(score, descending=True)
n = min(N_SHOW, len(order) // 2)
idx_tumor, idx_clean = order[:n].tolist(), order[-n:].tolist()

cov = brain.flatten(1).float().mean(1)
print(f"batch {tuple(x0.shape)}   cond union {UNION} -> {tuple(cond.shape)}   split by: {how}")
print(f"  brain coverage: min {float(cov.min()):.1%}  median {float(cov.median()):.1%} of the "
      f"slice  (lower CENTER_FRAC to narrow the band)")
print(f"  with tumor    {idx_tumor}   score {[round(float(score[i]), 3) for i in idx_tumor]}")
print(f"  without tumor {idx_clean}   score {[round(float(score[i]), 3) for i in idx_clean]}")

## Sample

One net at a time: load, run the reverse bridge on the shared batch, keep the recon, free it. That
keeps peak memory at one model regardless of how many runs are in the list.

In [ ]:
from models import build_model
from sb.base import build_schedule
from sb.i2sb import i2sb_sample


def apply_ema_(net, ema_path, device):
    '''Copy the saved EMA shadow into the model params, in the order the EMA class stored it.'''
    sd = torch.load(ema_path, map_location=device, weights_only=False)
    params = [p for p in net.parameters() if p.requires_grad]
    shadow = sd["shadow"]
    assert len(params) == len(shadow), f"EMA/param mismatch: {len(shadow)} vs {len(params)}"
    for p, s in zip(params, shadow):
        p.data.copy_(s.to(p.device))


recons = {}
for lbl, cfg_path in RUNS:
    cfg, ic = cfgs[lbl], cfgs[lbl]["i2sb"]
    save_dir = os.path.dirname(cfg_path)

    net = build_model(cfg).to(device)
    ck = torch.load(os.path.join(save_dir, "net.ckpt"), map_location=device, weights_only=False)
    net.load_state_dict(ck["model_state_dict"])
    net.eval()
    ema_path = os.path.join(save_dir, "ema.pt")
    used_ema = USE_EMA and os.path.exists(ema_path)
    if used_ema:
        apply_ema_(net, ema_path, device)

    bridge = build_schedule(kind=ic.get("kind", "brownian"), n_points=ic.get("n_points", 1000),
                            tau=ic.get("tau", 0.19), beta_max=ic.get("beta_max", 0.3),
                            device=device)
    # SBCDLNet / SBGroupCDL / SBUnet carry their own copy of the schedule and invert it to place
    # themselves on the bridge. If model.params drifted from cfg["i2sb"] they would sample on the
    # wrong coefficients -- silently, and the picture would just look bad for no visible reason.
    if hasattr(net, "assert_schedule_matches"):
        net.assert_schedule_matches(bridge)

    ci = cfg["data"][SPLIT].get("cond_idx", [])
    sel = [UNION.index(c) for c in ci]              # this net's channels, out of the shared stack
    c_in = cond[:, sel] if sel else None

    t0 = time.time()
    with torch.no_grad():
        rec, _, _ = i2sb_sample(
            net, x1, bridge, cond=c_in, nfe=NFE or ic.get("val_nfe", 20),
            deterministic=ic.get("deterministic", False),
            posterior=ic.get("posterior", "ddpm"),
            clip_denoise=ic.get("clip_denoise", False),
            target_channels=ic.get("target_channels", 1),
            log_count=1, verbose=False)
    recons[lbl] = rec.detach().cpu()

    print(f"  {lbl:<12} {cfg['model']['type']:<10} "
          f"{sum(p.numel() for p in net.parameters())/1e6:>6.2f}M  cond={ci}  "
          f"ema={'y' if used_ema else 'n'}  step={ck.get('step')}  {time.time()-t0:.1f}s")

    del net, bridge, rec
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

## Side by side

Two figures, same layout: prior, ground truth, then one column per method. Red contour is the ET
mask where the dataset has one.

In [ ]:
def panel(idx, title):
    if not idx:
        print(f"[{title}] nothing to show")
        return
    cols = [r"T1 (prior $x_1$)", r"T1ce (GT $x_0$)"] + [lbl for lbl, _ in RUNS]
    fig, ax = plt.subplots(len(idx), len(cols),
                           figsize=(2.45 * len(cols), 2.6 * len(idx)), squeeze=False)
    for r, b in enumerate(idx):
        m = mask[b, 0].cpu().numpy() > 0.5
        imgs = ([x1[b, 0].cpu().numpy(), x0[b, 0].cpu().numpy()]
                + [recons[lbl][b, 0].numpy() for lbl, _ in RUNS])
        for c, im in enumerate(imgs):
            # background to the window floor: 0 sits mid-grey on a signed scale and would read as
            # tissue
            ax[r][c].imshow(np.where(m, im, VMIN), cmap="gray", vmin=VMIN, vmax=VMAX)
            if et is not None:
                ax[r][c].contour(et[b, 0].cpu().numpy(), levels=[0.5],
                                 colors="r", linewidths=0.6)
            ax[r][c].set_xticks([]); ax[r][c].set_yticks([])
            if r == 0:
                ax[r][c].set_title(cols[c], fontsize=9)
        ax[r][0].set_ylabel(f"slice {b}", fontsize=8)
    fig.suptitle(f"{title}      fixed window [{VMIN:g}, {VMAX:g}], nfe={NFE}", fontsize=10)
    plt.tight_layout()
    plt.show()


panel(idx_tumor, f"WITH tumor  --  highest {len(idx_tumor)} by {how}")
panel(idx_clean, f"WITHOUT tumor  --  lowest {len(idx_clean)} by {how}")

## Stopping the bridge early

You do not need to change the sampler: `log_count=nfe` logs the state at **every** step, and
`xs[:, k]` is the state `k` steps before the end (`xs[:, 0]` is exactly the returned recon --
asserted below, because the trajectory is stored newest-first and that is easy to get backwards).

What the cell shows, per column, is the state at that stopping point together with the two
schedule numbers that say what is in it. The state on the bridge is

$$x_t = \mu_0 x_0 + \mu_1 x_1 + \sigma_{sb}\,\varepsilon$$

so stopping `k` steps early hands you an image containing $\mu_1$ of the **T1 prior** and
$\sigma_{sb}$ of **Gaussian noise**. Both grow as `k` grows. That extra high-frequency energy is
noise and leaked T1 anatomy -- it is not recovered T1ce detail, and it will read as texture.

Worth knowing before you tune this: with the DDPM posterior the final update is
`x_t <- a*x0_hat + b*x_t` with `a = 1 - sigma_p^2/sigma_n^2`, and at nfe=50 that is a = 0.95. The
output is therefore ~95% **one forward pass of the regressor at the lowest noise level**. If that
pass is smooth, the sampler cannot un-smooth it -- an MSE-trained x0 regressor predicts the
conditional mean, and a conditional mean is a blur. The fix for that is the loss, not the sampler
(`slurm/i2sb_vgg_sweep.sbatch`).

`a` does depend on nfe -- 0.99 at nfe=10, 0.95 at 50, 0.83 at 200, 0.67 at 500 -- so a higher nfe
does leave more of the running state (and its noise) in the output. That is a cheap thing to try
and costs nothing but sampling time.

In [ ]:
EARLY_RUN    = 0                  # index into RUNS
EARLY_STOPS  = [0, 1, 3, 10]      # steps before the end; 0 = the normal endpoint
EARLY_SLICES = 2                  # how many tumor-bearing slices to show

from sb.base import bridge_coeffs, space_indices, n_steps

lbl, cfg_path = RUNS[EARLY_RUN]
cfg, ic = cfgs[lbl], cfgs[lbl]["i2sb"]
save_dir = os.path.dirname(cfg_path)

net = build_model(cfg).to(device)
ck = torch.load(os.path.join(save_dir, "net.ckpt"), map_location=device, weights_only=False)
net.load_state_dict(ck["model_state_dict"])
net.eval()
if USE_EMA and os.path.exists(os.path.join(save_dir, "ema.pt")):
    apply_ema_(net, os.path.join(save_dir, "ema.pt"), device)

bridge = build_schedule(kind=ic.get("kind", "brownian"), n_points=ic.get("n_points", 1000),
                        tau=ic.get("tau", 0.19), beta_max=ic.get("beta_max", 0.3), device=device)
ci = cfg["data"][SPLIT].get("cond_idx", [])
c_in = cond[:, [UNION.index(c) for c in ci]] if ci else None

nfe_used = NFE or ic.get("val_nfe", 20)
with torch.no_grad():
    recon, xs, _ = i2sb_sample(net, x1, bridge, cond=c_in, nfe=nfe_used,
                               deterministic=ic.get("deterministic", False),
                               posterior=ic.get("posterior", "ddpm"),
                               clip_denoise=ic.get("clip_denoise", False),
                               target_channels=ic.get("target_channels", 1),
                               log_count=nfe_used, verbose=False)
# newest-first: index 0 is the endpoint, index k is k steps before it. Getting this backwards
# would silently plot the trajectory in reverse, so pin it.
assert torch.allclose(xs[:, 0], recon.cpu()), "trajectory is not newest-first"

mu0_t, mu1_t, sb_t = bridge_coeffs(bridge)
steps = space_indices(n_steps(bridge), nfe_used + 1)      # xs[:, k] lives at bridge step steps[k]
ks = [k for k in EARLY_STOPS if k < xs.shape[1]]

print(f"{lbl}: nfe={nfe_used}, {xs.shape[1]} logged steps")
print(f"{'k':>4} {'step':>6} {'mu1 (T1 leak)':>15} {'sigma_sb (noise)':>18}")
for k in ks:
    st = steps[k]
    print(f"{k:>4} {st:>6} {float(mu1_t[st]):>15.4f} {float(sb_t[st]):>18.4f}")

rows = idx_tumor[:EARLY_SLICES]
NL = chr(10)          # no backslash escapes here: this text survives two layers of
                      # string-literal processing on its way into the notebook
cols = ["T1ce (GT)"] + [f"stop k={k}{NL}mu1={float(mu1_t[steps[k]]):.3f}"
                        f"  sig_sb={float(sb_t[steps[k]]):.3f}" for k in ks]
fig, ax = plt.subplots(len(rows), len(cols),
                       figsize=(2.45 * len(cols), 2.7 * len(rows)), squeeze=False)
for r, b in enumerate(rows):
    m = mask[b, 0].cpu().numpy() > 0.5
    imgs = [x0[b, 0].cpu().numpy()] + [xs[b, k, 0].numpy() for k in ks]
    for c, im in enumerate(imgs):
        ax[r][c].imshow(np.where(m, im, VMIN), cmap="gray", vmin=VMIN, vmax=VMAX)
        if et is not None:
            ax[r][c].contour(et[b, 0].cpu().numpy(), levels=[0.5], colors="r", linewidths=0.6)
        ax[r][c].set_xticks([]); ax[r][c].set_yticks([])
        if r == 0:
            ax[r][c].set_title(cols[c], fontsize=8.5)
    ax[r][0].set_ylabel(f"slice {b}", fontsize=8)
fig.suptitle(f"{lbl}: stopping the bridge early -- the grain that appears as k "
             f"grows is sigma_sb noise and mu1 T1 leaking in, not recovered "
             f"T1ce detail.", fontsize=9)
plt.tight_layout(); plt.show()

del net, bridge
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()